# D006 국내 주식 시세 API 검증

## 목적
금융위원회 주식시세정보 API를 이용하여 국내 상장주식의 일별 시세 데이터를 조회할 수 있는지 검증한다.

### 테스트 대상
- 기업: 삼성전자
- 종목코드: 005930
- 시장: KOSPI

### 검증 항목
1. 최근 완료 거래일 20일 조회 가능 여부
2. 과거 특정 거래일 조회 가능 여부
3. 주요 OHLCV 필드 확인
4. 거래일 중복 여부 확인
5. 수정주가 제공 여부 확인

### 주요 필드
- 기준일자
- 종목코드
- 종목명
- 시장구분
- 시가
- 고가
- 저가
- 종가
- 거래량

> 주의: 본 API는 실시간 시세 API가 아니라 일별 시세 데이터 조회용이다.

In [ ]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)

In [ ]:
# 인증키가 Notebook에 저장되지 않도록 실행 시 직접 입력
SERVICE_KEY = getpass("공공데이터포털 Decoding 인증키 입력: ")

# 금융위원회 주식시세정보 V2
BASE_URL = (
    "https://apis.data.go.kr/1160100/"
    "GetStockSecuritiesInfoService_V2/getStockPriceInfo_V2"
)

# 테스트 종목
STOCK_CODE = "005930"
STOCK_NAME = "삼성전자"

In [ ]:
def get_stock_price(
    service_key,
    stock_code,
    begin_date=None,
    end_date=None,
    base_date=None,
    num_rows=100
):
    """
    금융위원회 주식시세정보 API 호출 함수

    Parameters
    ----------
    service_key : str
        공공데이터포털 인증키
    stock_code : str
        6자리 종목코드
    begin_date : str, optional
        조회 시작일 (YYYYMMDD)
    end_date : str, optional
        조회 종료일 (YYYYMMDD)
    base_date : str, optional
        특정 기준일 (YYYYMMDD)
    num_rows : int
        최대 조회 건수

    Returns
    -------
    pandas.DataFrame
    """

    params = {
        "serviceKey": service_key,
        "pageNo": 1,
        "numOfRows": num_rows,
        "resultType": "json",
        "likeSrtnCd": stock_code,
    }

    if begin_date is not None:
        params["beginBasDt"] = begin_date

    if end_date is not None:
        params["endBasDt"] = end_date

    if base_date is not None:
        params["basDt"] = base_date

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=(5, 60)
        )

        print("HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print("API 호출 실패")
            print(response.text[:1000])
            return pd.DataFrame()

        data = response.json()

        if "response" not in data:
            print("정상적인 API 응답 구조가 아닙니다.")
            print(data)
            return pd.DataFrame()

        header = data["response"]["header"]

        print(
            "API 결과:",
            header.get("resultCode"),
            header.get("resultMsg")
        )

        body = data["response"].get("body", {})
        items = body.get("items", {})

        if not items:
            print("조회 결과가 없습니다.")
            return pd.DataFrame()

        rows = items.get("item", [])

        if isinstance(rows, dict):
            rows = [rows]

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과(ReadTimeout)")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

## 1. 최근 20거래일 조회

최근 약 60일의 데이터를 요청한 후 실제 거래일 기준 최신 20건을 추출한다.

In [ ]:
today = pd.Timestamp.today().normalize()

begin_date = (
    today - pd.Timedelta(days=60)
).strftime("%Y%m%d")

end_date = (
    today + pd.Timedelta(days=1)
).strftime("%Y%m%d")

df = get_stock_price(
    service_key=SERVICE_KEY,
    stock_code=STOCK_CODE,
    begin_date=begin_date,
    end_date=end_date,
    num_rows=100
)

print("전체 조회 건수:", len(df))

display(df.head())

In [ ]:
if not df.empty:

    # 정확히 삼성전자만 필터링
    df = df[
        df["srtnCd"].astype(str).str.zfill(6) == STOCK_CODE
    ].copy()

    # 날짜 변환
    df["basDt"] = pd.to_datetime(
        df["basDt"],
        format="%Y%m%d",
        errors="coerce"
    )

    # 숫자형 변환
    numeric_cols = [
        "mkp",          # 시가
        "hipr",         # 고가
        "lopr",         # 저가
        "clpr",         # 종가
        "trqu",         # 거래량
        "trPrc",        # 거래대금
        "lstgStCnt",    # 상장주식수
        "mrktTotAmt"    # 시가총액
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # 최신 거래일부터 정렬
    df = df.sort_values(
        "basDt",
        ascending=False
    ).reset_index(drop=True)

In [ ]:
recent20 = df.head(20)[
    [
        "basDt",
        "srtnCd",
        "itmsNm",
        "mrktCtg",
        "mkp",
        "hipr",
        "lopr",
        "clpr",
        "trqu"
    ]
].copy()

recent20 = recent20.rename(columns={
    "basDt": "날짜",
    "srtnCd": "종목코드",
    "itmsNm": "종목명",
    "mrktCtg": "시장",
    "mkp": "시가",
    "hipr": "고가",
    "lopr": "저가",
    "clpr": "종가",
    "trqu": "거래량"
})

display(recent20)

print("최근 거래일 수:", len(recent20))

In [ ]:
## 2. 과거 특정 거래일 조회

과거 특정 날짜의 삼성전자 시세를 조회하여 장기 시계열 접근 가능 여부를 확인한다.

테스트 일자: 2025-01-02

In [ ]:
TEST_DATE = "20250102"

old_df = get_stock_price(
    service_key=SERVICE_KEY,
    stock_code=STOCK_CODE,
    base_date=TEST_DATE,
    num_rows=10
)

if not old_df.empty:

    old_result = old_df[
        [
            "basDt",
            "srtnCd",
            "itmsNm",
            "mrktCtg",
            "mkp",
            "hipr",
            "lopr",
            "clpr",
            "trqu"
        ]
    ].copy()

    display(old_result)

In [ ]:
## 3. 기본 데이터 품질 검증

최근 20거래일 데이터에 중복 날짜 및 주요 값 결측이 존재하는지 확인한다.

In [ ]:
print("===== D006 데이터 품질 검증 =====")

print("행 개수:", len(recent20))
print("고유 날짜 개수:", recent20["날짜"].nunique())
print("중복 날짜 수:", recent20["날짜"].duplicated().sum())

print("\n주요 컬럼 결측치:")
display(
    recent20[
        ["날짜", "시가", "고가", "저가", "종가", "거래량"]
    ].isna().sum().to_frame("결측 개수")
)

In [ ]:
print("최근 20거래일:")

display(
    recent20[
        ["날짜", "종가", "거래량"]
    ].sort_values("날짜")
)

In [ ]:
## 4. 검증 결과

### D006 국내 주식 시세 API

| 검증 항목 | 결과 |
|---|---|
| API 활용신청 | 완료 |
| API 연결 | 성공 |
| 삼성전자 005930 조회 | 성공 |
| 최근 20거래일 조회 | 성공 |
| 과거 특정일 조회 | 성공 |
| 시가·고가·저가·종가 | 확인 |
| 거래량 | 확인 |
| 거래일 중복 확인 | 가능 |
| 수정주가 여부 | 추가 검증 필요 |

### 확인된 주요 필드

- `basDt` : 기준일자
- `srtnCd` : 종목코드
- `isinCd` : ISIN
- `itmsNm` : 종목명
- `mrktCtg` : 시장구분
- `mkp` : 시가
- `hipr` : 고가
- `lopr` : 저가
- `clpr` : 종가
- `trqu` : 거래량
- `trPrc` : 거래대금
- `lstgStCnt` : 상장주식수
- `mrktTotAmt` : 시가총액

### 주의사항

1. 본 API는 실시간 주가 API가 아닌 일별 시세 데이터 조회용이다.
2. 별도의 수정종가(adjusted close) 필드가 확인되지 않아 수정주가 여부는 추가 검증이 필요하다.
3. 주말·공휴일 등 비거래일에 데이터가 존재하지 않는 것은 정상이다.
4. API 인증키는 Notebook에 직접 저장하지 않고 실행 시 입력하도록 구성하였다.

In [ ]:
print("===== D006 최종 검증 결과 =====")
print(f"테스트 종목: {STOCK_NAME} ({STOCK_CODE})")
print(f"최근 20거래일 확보: {len(recent20) == 20}")
print(f"최근 데이터 중복: {recent20['날짜'].duplicated().sum()}")
print(f"과거 특정일 조회: {not old_df.empty}")
print("수정주가 여부: 별도 검증 필요")